# Backtest Statistics


Strategy: **sentiment_asynchronous**.

## Account Simulation

- **Purpose:** Evaluate the single self-financing cross-sectional meta-filtered holdout account.
- **Settings:** [Shared portfolio settings](portfolio_management.ipynb); SPY benchmark; annual risk-free = 3%; benchmark Sharpe = 1; cash interest/borrow fees = 0.
- **Data:** Saved predictions, price calibration, market observations, account ledger, fills, and closed position episodes.
- **Findings:**
- **Decision:** Simulate once and use the same account state for every investment statistic.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from src.backtesting.portfolio_management import PortfolioSettings, run_final_backtest
from src.preprocessing.market_data import ResearchPaths

PROJECT_ROOT = Path.cwd().resolve().parents[2]
PERIOD = "2025-02-01_2025-12-31"
MANIFEST_PATH = PROJECT_ROOT / "data/preprocessing/universe/sp500_2025.csv"
EXPECTED_SECURITIES = 503
DATA_START = pd.Timestamp("2025-01-01", tz="UTC")
END = pd.Timestamp("2026-01-01", tz="UTC")
HORIZON_BARS = 1000
K = 10
STEP_SIZE = 0.10
INITIAL_AUM = 100_000.0
BROKER_FEE_BPS = 1.0
SLIPPAGE_BPS = 1.0
HALF_LIFE_MINUTES = 390.0
PERIODS_PER_YEAR = 365.25
paths = ResearchPaths(PROJECT_ROOT, model_kind="sentiment", period=PERIOD)
STRATEGY = "sentiment_asynchronous"
settings = PortfolioSettings(
    k=K, step_size=STEP_SIZE, initial_aum=INITIAL_AUM,
    broker_fee_bps=BROKER_FEE_BPS, slippage_bps=SLIPPAGE_BPS,
    half_life_minutes=HALF_LIFE_MINUTES,
)
annual_risk_free_rate = 0.03
annualized_benchmark_sharpe_ratio = 1.0
statistics, result = run_final_backtest(
    paths,
    settings,
    strategy=STRATEGY,
    manifest_path=MANIFEST_PATH,
    expected_securities=EXPECTED_SECURITIES,
    data_start=DATA_START,
    end=END,
    horizon_bars=HORIZON_BARS,
    annual_risk_free_rate=annual_risk_free_rate,
    periods_per_year=PERIODS_PER_YEAR,
    annualized_benchmark_sharpe_ratio=annualized_benchmark_sharpe_ratio,
)


def statistics_table(section):
    selected = statistics.loc[statistics["section"].eq(section)].copy()
    selected["value"] = selected["value"].where(
        selected["unit"].ne("timestamp"), selected["timestamp"]
    )
    return selected.pivot(index="entity", columns="metric", values="value").reindex(
        index=selected["entity"].drop_duplicates(),
        columns=selected["metric"].drop_duplicates(),
    ).rename_axis(columns=None).infer_objects()


display(result["exclusions"])


## General Characteristics

- **Purpose:** Describe account span, capital, exposure, trade frequency, holding time, turnover, and benchmark relationship.
- **Settings:** Closed positions are flat-to-flat episodes; same-direction resizing stays in one episode; turnover uses 365.25-day years.
- **Data:** Calendar-day AUM, full-resolution symbol exposure, incremental traded notional, and SPY daily returns.
- **Findings:**
- **Decision:** Flat accounts report zero bet frequency and undefined direction or holding-period statistics.

In [ ]:
general_characteristics = statistics_table("general_characteristics")
display(general_characteristics)


## Performance

- **Purpose:** Report account PnL, elapsed-time annualized return, and closed-position outcomes.
- **Settings:** Annualize over elapsed 365.25-day years; attribute long/short PnL by each episode’s opening side.
- **Data:** Final account AUM and reconciled closed-position PnL and returns.
- **Findings:**
- **Decision:** Dollar PnL is not inferred by summing overlapping event returns.

In [ ]:
performance = statistics_table("performance").reindex(columns=[
    "pnl", "pnl_from_long_positions", "pnl_from_short_positions",
    "annualized_rate_of_return", "hit_ratio",
    "average_return_from_hits", "average_return_from_misses",
])
display(performance)


## Runs

- **Purpose:** Measure return and timing concentration together with drawdown depth and duration.
- **Settings:** HHI uses closed-position returns and entry months; episode percentiles use 95%.
- **Data:** The equity path includes initial capital and every retained account observation.
- **Findings:**
- **Decision:** Undefined concentration and no-drawdown percentiles remain NaN.

In [ ]:
runs = statistics_table("runs")
display(runs)


## Implementation Shortfall

- **Purpose:** Relate broker fees, slippage, and net performance to actual traded notional.
- **Settings:** Cumulative ledger counters are differenced before aggregation.
- **Data:** All entries, resizing trades, replacements, reductions, and final liquidations.
- **Findings:**
- **Decision:** Dollar performance is net of broker fees and adverse execution slippage.

In [ ]:
implementation_shortfall = statistics_table("implementation_shortfall")
display(implementation_shortfall)


## Efficiency

- **Purpose:** Evaluate return per unit of variability and the probability that annualized Sharpe exceeds one.
- **Settings:** Calendar-day observations, 365.25 annualization, and 3% effective annual risk-free return over actual intervals.
- **Data:** Full-account returns; the first period includes opening costs relative to initial capital.
- **Findings:**
- **Decision:** Cash earns no interest in the simulated account, while the risk-free rate remains the evaluation benchmark.

In [ ]:
efficiency = statistics_table("efficiency")
display(efficiency)


## Classification Scores

- **Purpose:** Evaluate the frozen primary direction and meta action classifiers on holdout events.
- **Settings:** Stored sample weights, positive label 1, and complete probability matrices.
- **Data:** Primary labels {-1, 1} and meta labels {0, 1} from the saved prediction artifact.
- **Findings:**
- **Decision:** Report accuracy, precision, recall, F1, and negative log loss for both models.

In [ ]:
classification_scores = statistics_table("classification_scores").T.rename_axis(
    index=None, columns=None
)
display(classification_scores)


## Final Result

- **Purpose:** Persist strategy-specific statistics and account curves containing every statistics family shown above.
- **Settings:** Numeric values and UTC timestamps use separate typed columns.
- **Data:** Account, benchmark, exposure, classification, assumption, and coverage results.
- **Findings:**
- **Decision:** Recomputing the artifact requires rerunning the final account simulation and this ordered review.

In [ ]:
required_sections = {
    "general_characteristics",
    "performance",
    "runs",
    "implementation_shortfall",
    "efficiency",
    "classification_scores",
}
if not required_sections.issubset(set(statistics["section"])):
    raise ValueError("Final statistics are missing a required evaluation family")
result_path = PROJECT_ROOT / "data/backtest_results/sentiment_asynchronous/backtest_statistics.parquet"
display(statistics)
print(result_path)